In [1]:
import pandas as pd
import re
import csv

.CSV obtained from de UKB database (date/source) 

Load data

In [2]:
df = pd.read_csv("../data/ukb_simulated_data/labels.csv", header=None, sep="\t")
df_complete = pd.read_csv('../data/UKB_complete_list.tsv', sep='\t')

In [3]:
# Delphi's disease data 
df

,0
0,Padding
1,No event
2,Female
3,Male
4,BMI_low
...,...
1265,D46 Myelodysplastic syndromes
1266,D47 Other neoplasms of uncertain or unknown be...
1267,D48 Neoplasm of uncertain or unknown behaviour...
1268,O01 Hydatidiform mole38


In [4]:
# ukb complete disease list
df_complete

,coding,meaning,node_id,parent_id,selectable
0,A00,A00 Cholera,2860,230,Y
1,A000,"A00.0 Cholera due to Vibrio cholerae 01, biova...",2870,2860,Y
2,A001,"A00.1 Cholera due to Vibrio cholerae 01, biova...",2880,2860,Y
3,A009,"A00.9 Cholera, unspecified",2890,2860,Y
4,A01,A01 Typhoid and paratyphoid fevers,2900,230,Y
...,...,...,...,...,...
19185,Z992,Z99.2 Dependence on renal dialysis,191500,191470,Y
19186,Z993,Z99.3 Dependence on wheelchair,191510,191470,Y
19187,Z994,Z99.4 Dependence on artificial heart,191520,191470,Y
19188,Z998,Z99.8 Dependence on other enabling machines an...,191530,191470,Y


In [5]:
# separate code disease from text
df.columns = ["raw"]

extracted = df["raw"].str.extract(r"^([A-Z]\d+)\s+(.+)$")
df["code"]        = extracted[0]
df["description"] = extracted[1].fillna(df["raw"])  # exposome data

df = df.drop(columns="raw")

In [6]:
df

,code,description
0,NaN,Padding
1,NaN,No event
2,NaN,Female
3,NaN,Male
4,NaN,BMI_low
...,...,...
1265,D46,Myelodysplastic syndromes
1266,D47,Other neoplasms of uncertain or unknown behavi...
1267,D48,Neoplasm of uncertain or unknown behaviour of ...
1268,O01,Hydatidiform mole38


In [7]:
# check which codes from df are in df_complete and delete the rest 
df_diseases = df.merge(
    df_complete[['coding', 'meaning']],
    left_on='code', right_on='coding',
    how='left'
)[['code', 'description', 'meaning']]

In [8]:
df_diseases

,code,description,meaning
0,NaN,Padding,NaN
1,NaN,No event,NaN
2,NaN,Female,NaN
3,NaN,Male,NaN
4,NaN,BMI_low,NaN
...,...,...,...
1265,D46,Myelodysplastic syndromes,D46 Myelodysplastic syndromes
1266,D47,Other neoplasms of uncertain or unknown behavi...,D47 Other neoplasms of uncertain or unknown be...
1267,D48,Neoplasm of uncertain or unknown behaviour of ...,D48 Neoplasm of uncertain or unknown behaviour...
1268,O01,Hydatidiform mole38,O01 Hydatidiform mole


In [9]:
# delete de code from the meaning column
mask = df_diseases['meaning'].notna()
df_diseases.loc[mask, 'meaning'] = df_diseases.loc[mask, 'meaning'].str.replace(r'^[A-Z]\d+[\.\d]*\s+', '', regex=True)

In [10]:
df_diseases 

,code,description,meaning
0,NaN,Padding,NaN
1,NaN,No event,NaN
2,NaN,Female,NaN
3,NaN,Male,NaN
4,NaN,BMI_low,NaN
...,...,...,...
1265,D46,Myelodysplastic syndromes,Myelodysplastic syndromes
1266,D47,Other neoplasms of uncertain or unknown behavi...,Other neoplasms of uncertain or unknown behavi...
1267,D48,Neoplasm of uncertain or unknown behaviour of ...,Neoplasm of uncertain or unknown behaviour of ...
1268,O01,Hydatidiform mole38,Hydatidiform mole


In [11]:
# fill the meaning column with the description if the meaning is NaN
df_diseases["meaning"] = df_diseases["meaning"].fillna(df_diseases["description"])

In [12]:
df_diseases

,code,description,meaning
0,NaN,Padding,Padding
1,NaN,No event,No event
2,NaN,Female,Female
3,NaN,Male,Male
4,NaN,BMI_low,BMI_low
...,...,...,...
1265,D46,Myelodysplastic syndromes,Myelodysplastic syndromes
1266,D47,Other neoplasms of uncertain or unknown behavi...,Other neoplasms of uncertain or unknown behavi...
1267,D48,Neoplasm of uncertain or unknown behaviour of ...,Neoplasm of uncertain or unknown behaviour of ...
1268,O01,Hydatidiform mole38,Hydatidiform mole


In [13]:
# save to .csv for generating the embeddings
df_diseases.to_csv("../data/labels_filtered.csv", index=False)